In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rsync -a /content/drive/MyDrive/NUDGE_patched/ /content/NUDGE/
%cd /content/NUDGE
!pip uninstall -y -q tensorflow jax jaxlib
!pip install -q -e nsfr -e nudge
!rm -rf out && ln -s /content/drive/MyDrive/NUDGE_out out

In [1]:
!git clone https://github.com/k4ntz/NUDGE.git
%cd NUDGE
!pip install -r requirements.txt

Cloning into 'NUDGE'...
remote: Enumerating objects: 1982, done.
remote: Counting objects: 100% (323/323), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 1982 (delta 198), reused 226 (delta 168), pack-reused 1659 (from 2)
Receiving objects: 100% (1982/1982), 231.96 MiB | 20.06 MiB/s, done.
Resolving deltas: 100% (596/596), done.
Updating files: 100% (1582/1582), done.
/content/NUDGE
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 138.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  Getti

In [26]:
%cd /content/NUDGE
!python train.py -g getout

/content/NUDGE
Hyperparameter Summary:
algorithm: logic
device: cpu
env_kwargs: {}
environment: getout
epochs: 20
eps_clip: 0.2
epsilon_fn: !!python/name:nudge.utils.exp_decay ''
gamma: 0.99
lr_actor: 0.001
lr_critic: 0.0003
max_ep_len: 500
optimizer: !!python/name:torch.optim.adam.Adam ''
recover: false
rules: getout_bs_top10
save_steps: 50000
seed: 0
stats_steps: 2000
total_steps: 800000
update_steps: 100

C_0:  0.07 jump(X):-closeby(O1,O2),obj_type(O1,agent),obj_type(O2,enemy),on_left(O1,O2).
C_1:  0.07 jump(X):-closeby(O1,O2),obj_type(O1,agent),obj_type(O2,enemy),on_right(O2,O1).
C_2:  0.05 jump(X):-closeby(O1,O2),obj_type(O1,agent),obj_type(O2,enemy).
C_3:  0.09 jump(X):-have_key(X),obj_type(O1,agent),obj_type(O2,enemy),on_left(O1,O2).
C_4:  0.06 jump(X):-have_key(X),obj_type(O1,agent),obj_type(O2,enemy),on_left(O2,O1).
C_5:  0.06 jump(X):-have_key(X),obj_type(O1,agent),obj_type(O2,enemy),on_right(O1,O2).
C_6:  0.04 jump(X):-have_key(X),obj_type(O1,agent),obj_type(O2,enemy),on_rig

In [25]:
!sed -i 's/^total_steps:.*/total_steps: 400000/' in/config/default.yaml
!grep -n "total_steps\|save_steps\|device" in/config/default.yaml

5:device: cpu
6:save_steps: 50000


In [19]:
!sed -i 's/^device:.*/device: cpu/' in/config/default.yaml
!grep -n "device\|save_steps" in/config/default.yaml

5:device: cpu
6:save_steps: 50000


In [21]:
from google.colab import drive
drive.mount('/content/drive')
!rsync -a --delete /content/NUDGE/ /content/drive/MyDrive/NUDGE_patched/
!ls /content/drive/MyDrive/NUDGE_patched | head

Mounted at /content/drive
docs
env_src
in
__init__.py
LICENSE
nsfr
nudge
out
play_gui.py
play.py


In [22]:
!grep -n "out\|save" train.py | head -20

20:from nudge.utils import make_deterministic, save_hyperparams
27:OUT_PATH = Path("out/")
48:         save_steps: int = 250000,
75:        save_steps: Number of steps between each checkpoint save
102:    save_hyperparams(signature=signature(main),
104:                     save_path=experiment_dir / "config.yaml",
149:    pbar = tqdm(total=total_steps - time_step, file=sys.stdout)
188:            # save model weights
189:            if time_step % save_steps == 1:
192:                    agent.save(checkpoint_path, checkpoint_dir, step_list, reward_list, weights_list)
194:                    agent.save(checkpoint_path, checkpoint_dir, step_list, reward_list)


In [23]:
!mkdir -p /content/drive/MyDrive/NUDGE_out
!rm -rf /content/NUDGE/out && ln -s /content/drive/MyDrive/NUDGE_out /content/NUDGE/out

In [14]:
!sed -i 's/^device:.*/device: cuda/' in/config/default.yaml

In [16]:
!grep -n "device\|save_steps" in/config/default.yaml

In [17]:
!echo "device: cuda" >> in/config/default.yaml
!echo "save_steps: 50000" >> in/config/default.yaml

In [10]:
# 1. remove the ipdb breakpoint
!sed -i 's/import ipdb; ipdb.set_trace()/raise NotImplementedError/' env_src/getout/getout/getout.py

# 2. type -> obj_type in the rule set only
!sed -i 's/\btype(/obj_type(/g' in/envs/getout/logic/getout_bs_top10/clauses.txt
!sed -i 's/^type:/obj_type:/' in/envs/getout/logic/getout_bs_top10/neural_preds.txt

# 3. working reset/step, attached to NudgeEnv
patch = '''

def _nudge_reset(self):
    import random
    from env_src.getout.getout.getout import Getout
    from env_src.getout.getout.paramLevelGenerator import ParameterizedLevelGenerator
    self.env = Getout()
    gen = ParameterizedLevelGenerator(enemy=True, enemies=self.plusplus, key_door=True)
    gen.generate(self.env, seed=random.randint(0, 2**31 - 1))
    return self.convert_state(self.env)

def _nudge_step(self, action, is_mapped: bool = False):
    from env_src.getout.getout.actions import GetoutActions
    if isinstance(action, str):
        action = self.pred2action.get(action.split("_")[0], 0)
    action = int(action)
    names = {m.name for m in GetoutActions}
    jump = "MOVE_UP" if "MOVE_UP" in names else "JUMP"
    mapping = {0: [GetoutActions.NOOP], 1: [GetoutActions.MOVE_LEFT],
               2: [GetoutActions.MOVE_RIGHT], 3: [getattr(GetoutActions, jump)]}
    out = self.env.step(mapping.get(action, [GetoutActions.NOOP]))
    if isinstance(out, tuple):
        state, reward, done = out[0], out[1], out[2]
    else:
        state, reward, done = self.env, (out or 0), self.env.level.terminated
    return self.convert_state(state), reward, done

NudgeEnv.reset = _nudge_reset
NudgeEnv.step = _nudge_step
'''
open("in/envs/getout/env.py", "a").write(patch)

!grep -c "obj_type(" in/envs/getout/logic/getout_bs_top10/clauses.txt
!cat in/envs/getout/logic/getout_bs_top10/neural_preds.txt

50
obj_type:2:object,type
closeby:2:object,object
on_left:2:object,object
on_right:2:object,object
have_key:1:image
not_have_key:1:image


In [7]:
!sed -i 's/^\(\s*rules:\).*/\1 getout_bs_top10/' in/config/getout.yaml
!grep -n "rules" in/config/getout.yaml

sed: can't read in/config/getout.yaml: No such file or directory
grep: in/config/getout.yaml: No such file or directory


In [8]:
!sed -i 's/^\(\s*rules:\).*/\1 getout_bs_top10/' in/config/default.yaml
!grep -rn "rules" in/config/

in/config/debug.yaml:5:rules: default
in/config/default.yaml:3:rules: getout_bs_top10
in/config/loot.yaml:3:rules: default


In [8]:
!ls in/envs/getout/logic/
!grep -n "rules" in/config/default.yaml

getout_bs_rf1	 getout_human_assisted	getout_no_search_5	  getout_root
getout_bs_rf3	 getout_kd		getout_no_search_50
getout_bs_top10  getout_no_search	getoutplus
getout_e	 getout_no_search_15	getout_redundant_actions
3:rules: default


In [9]:
!ls in/envs/getout/logic/getout_bs_top10
!cp in/config/default.yaml in/config/getout.yaml
!sed -i 's/^rules: default/rules: getout_bs_top10/' in/config/getout.yaml
!grep -n "rules" in/config/getout.yaml
!python train.py -c in/config/getout.yaml -g getout

bk.txt	clauses_50.txt	clauses.txt  consts.txt  neural_preds.txt  preds.txt
3:rules: getout_bs_top10
Hyperparameter Summary:
algorithm: logic
device: cpu
env_kwargs: {}
environment: getout
epochs: 20
eps_clip: 0.2
epsilon_fn: !!python/name:nudge.utils.exp_decay ''
gamma: 0.99
lr_actor: 0.001
lr_critic: 0.0003
max_ep_len: 500
optimizer: !!python/name:torch.optim.adam.Adam ''
recover: false
rules: getout_bs_top10
save_steps: 250000
seed: 0
stats_steps: 2000
total_steps: 800000
update_steps: 100

C_0:  0.07 jump(X):-closeby(O1,O2),on_left(O1,O2),type(O1,agent),type(O2,enemy).
C_1:  0.07 jump(X):-closeby(O1,O2),on_right(O2,O1),type(O1,agent),type(O2,enemy).
C_2:  0.05 jump(X):-closeby(O1,O2),type(O1,agent),type(O2,enemy).
C_3:  0.09 jump(X):-have_key(X),on_left(O1,O2),type(O1,agent),type(O2,enemy).
C_4:  0.06 jump(X):-have_key(X),on_left(O2,O1),type(O1,agent),type(O2,enemy).
C_5:  0.06 jump(X):-have_key(X),on_right(O1,O2),type(O1,agent),type(O2,enemy).
C_6:  0.04 jump(X):-have_key(X),on_rig

In [10]:
!sed -i 's/import ipdb; ipdb.set_trace()/raise/' env_src/getout/getout/getout.py
!sed -n 80,95p env_src/getout/getout/getout.py
!python train.py -c in/config/getout.yaml -g getout 2>&1 | tail -25

        if self.camera is not None:
            self.camera.start_render()
            self.level.render(self.camera, self.step_counter)
            self.camera.end_render()

            self.hud.render(self.camera, step=self.step_counter if self.show_step_counter else None)

    def get_score(self):
        return self.score

    def reset(self):
        raise
        self.step_counter = 0
        return [0 for _ in range(8)]

C_38:  0.03 right_go_get_key(X):-not_have_key(X),on_right(O2,O1),type(O1,agent),type(O2,key).
C_39:  0.06 right_go_get_key(X):-not_have_key(X),type(O1,agent),type(O2,key).
C_40:  0.02 right_go_to_door(X):-closeby(O1,O2),have_key(X),type(O1,agent),type(O2,door).
C_41:  0.05 right_go_to_door(X):-closeby(O1,O2),on_left(O1,O2),type(O1,agent),type(O2,door).
C_42:  0.03 right_go_to_door(X):-closeby(O1,O2),on_right(O2,O1),type(O1,agent),type(O2,door).
C_43:  0.07 right_go_to_door(X):-closeby(O1,O2),type(O1,agent),type(O2,door).
C_44:  0.1 right_go_to_door(X):-have_key(

In [11]:
!cat in/envs/getout/env.py
!grep -n "class \|def \|import" env_src/getout/getout/paramLevelGenerator.py

from enum import Enum
from typing import Sequence

from nudge.env import NudgeBaseEnv
import numpy as np
import gymnasium
import gym3
from gymnasium.envs.registration import register
from nudge.utils import simulate_prob
import torch


class NudgeEnv(NudgeBaseEnv):
    name = "getout"
    pred2action = {
        'stay': 0,
        'idle': 0,
        'left': 1,
        'right': 2,
        'jump': 3,
    }
    pred_names: Sequence

    def __init__(self, mode: str, plusplus=False, noise=False):
        super().__init__(mode)
        self.plusplus = plusplus
        self.noise = noise
        register(id="getout",
                 entry_point="env_src.getout.getout.getout:Getout")
        self.env = gymnasium.envs.registration.load_env_creator(gymnasium.spec("getout").entry_point)(**gymnasium.spec("getout").kwargs)  # FIXME
        # self.env = gym3.make("getout")  # FIXME

    def reset(self):
        state = self.env.reset()
        return self.convert_state(state)

    def step(self, a

In [12]:
patch = r'''
    def reset(self):
        import random
        from env_src.getout.getout.getout import Getout
        from env_src.getout.getout.paramLevelGenerator import ParameterizedLevelGenerator
        self.env = Getout()
        gen = ParameterizedLevelGenerator(enemy=True, enemies=self.plusplus, key_door=True)
        gen.generate(self.env, seed=random.randint(0, 2**31 - 1))
        return self.convert_state(self.env)

    def step(self, action, is_mapped: bool = False):
        out = self.env.step(action)
        if isinstance(out, tuple):
            state, reward, done = out[0], out[1], out[2]
        else:
            state, reward, done = self.env, out, self.env.level.terminated
        return self.convert_state(state), reward, done
'''
src = open("in/envs/getout/env.py").read()
start = src.index("    def reset(self):")
end = src.index("    def extract_logic_state")
open("in/envs/getout/env.py", "w").write(src[:start] + patch.lstrip("\n") + "\n" + src[end:])
print("patched")

patched


In [13]:
!grep -n "def step" -A 12 env_src/getout/getout/getout.py
!python train.py -c in/config/getout.yaml -g getout 2>&1 | tail -25

48:    def step(self, action):
49-        if self.level.terminated:
50-            return 0
51-
52-        # start stepping the game only after the first action.
53-        if self.start_on_first_action and not self.has_started:
54-            if isinstance(action, int):
55-                la = 0
56-            else:
57-                la = len(action)
58-            if la == 0 or (la == 1 and action[0] == GetoutActions.NOOP):
59-                return None
60-            else:
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1794, in _call_impl
    return forward_call(*args, **kwargs)
  File "/content/NUDGE/nsfr/nsfr/nsfr.py", line 42, in forward
    self.V_0 = self.fc(zs, self.atoms, self.bk)
               ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1783, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/loca

In [14]:
import os, re
vfile = "in/envs/getout/valuation.py"
funcs = set(re.findall(r"^def (\w+)\(", open(vfile).read(), re.M))
print("valuation functions:", sorted(funcs), "\n")

root = "in/envs/getout/logic"
for rs in sorted(os.listdir(root)):
    path = os.path.join(root, rs)
    used = set()
    for f in os.listdir(path):
        if f.endswith(".txt"):
            used |= set(re.findall(r"(\w+)\(", open(os.path.join(path, f)).read()))
    p = os.path.join(path, "preds.txt")
    preds = set(re.findall(r"^(\w+):", open(p).read(), re.M)) if os.path.exists(p) else set()
    missing = sorted((used - preds) - funcs)   # predicates in rules that aren't actions and have no function
    print(f"{rs:28s} missing: {missing if missing else 'none'}")

valuation functions: ['closeby', 'have_key', 'not_have_key', 'obj_type', 'on_left', 'on_right', 'safe'] 

getout_bs_rf1                missing: ['type']
getout_bs_rf3                missing: ['type']
getout_bs_top10              missing: ['type']
getout_e                     missing: ['type']
getout_human_assisted        missing: ['type']
getout_kd                    missing: ['type']
getout_no_search             missing: ['type']
getout_no_search_15          missing: ['type']
getout_no_search_5           missing: ['type']
getout_no_search_50          missing: ['type']
getout_redundant_actions     missing: ['type']
getout_root                  missing: ['type']
getoutplus                   missing: ['type']


In [15]:
!sed -i -E 's/\btype\b/obj_type/g' in/envs/getout/logic/getout_bs_top10/*.txt
!grep -c "obj_type" in/envs/getout/logic/getout_bs_top10/*.txt
!sed -i 's/^rules: .*/rules: getout_bs_top10/' in/config/getout.yaml
!python train.py -c in/config/getout.yaml -g getout 2>&1 | tail -25

in/envs/getout/logic/getout_bs_top10/bk.txt:0
in/envs/getout/logic/getout_bs_top10/clauses_50.txt:50
in/envs/getout/logic/getout_bs_top10/clauses.txt:50
in/envs/getout/logic/getout_bs_top10/consts.txt:1
in/envs/getout/logic/getout_bs_top10/neural_preds.txt:1
in/envs/getout/logic/getout_bs_top10/preds.txt:0
    self.V_0 = self.fc(zs, self.atoms, self.bk)
               ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1783, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1794, in _call_impl
    return forward_call(*args, **kwargs)
  File "/content/NUDGE/nsfr/nsfr/facts_converter.py", line 29, in forward
    return self.convert(Z, G, B)
           ~~~~~~~~~~~~^^^^^^^^^
  File "/content/NUDGE/nsfr/nsfr/facts_converter.py", line 60, in convert
    V[:, i] = self.vm(Z, atom)
          

In [16]:
!sed -i -E 's/\bobj_type\b/type/g' in/envs/getout/logic/getout_bs_top10/consts.txt
!sed -i -E 's/([:,])obj_type\b/\1type/g' in/envs/getout/logic/getout_bs_top10/neural_preds.txt
!cat in/envs/getout/logic/getout_bs_top10/consts.txt
!cat in/envs/getout/logic/getout_bs_top10/neural_preds.txt
!python train.py -c in/config/getout.yaml -g getout 2>&1 | tail -25

object:obj1,obj2,obj3,obj4
type:agent,key,door,enemy
image:imgobj_type:2:object,type
closeby:2:object,object
on_left:2:object,object
on_right:2:object,object
have_key:1:image
not_have_key:1:image
C_48:  0.11 right_go_to_door(X):-not_have_key(X),obj_type(O1,agent),obj_type(O2,door),on_right(O2,O1).
C_49:  0.05 right_go_to_door(X):-not_have_key(X),obj_type(O1,agent),obj_type(O2,door).
Started training at  19:13
  0%|          | 0/800000 [00:00<?, ?it/s]/content/NUDGE/nudge/agents/logic_agent.py:89: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  neural_state = torch.tensor(neural_state, dtype=torch.float32, device=self.device).unsqueeze(0)
Traceback (most recent call last):
  File "/content/NUDGE/train.py", line 249, in <module>
    main(**config)
    ~~~~^^^^^^^^^^
  File "/content/NUDGE/train.py", line 160, in main
    state, reward, done

In [17]:
patch = r'''
    def step(self, action, is_mapped: bool = False):
        from env_src.getout.getout.actions import GetoutActions
        if isinstance(action, str):                       # e.g. "right" or "right_go_to_door"
            action = self.pred2action.get(action.split("_")[0], 0)
        action = int(action)
        names = {m.name for m in GetoutActions}
        jump = "MOVE_UP" if "MOVE_UP" in names else "JUMP"
        mapping = {0: [GetoutActions.NOOP], 1: [GetoutActions.MOVE_LEFT],
                   2: [GetoutActions.MOVE_RIGHT], 3: [getattr(GetoutActions, jump)]}
        out = self.env.step(mapping.get(action, [GetoutActions.NOOP]))
        if isinstance(out, tuple):
            state, reward, done = out[0], out[1], out[2]
        else:
            state, reward, done = self.env, (out or 0), self.env.level.terminated
        return self.convert_state(state), reward, done
'''
src = open("in/envs/getout/env.py").read()
start = src.index("    def step(self, action")
end = src.index("    def extract_logic_state")
open("in/envs/getout/env.py", "w").write(src[:start] + patch.lstrip("\n") + "\n" + src[end:])
print("patched")

patched


In [ ]:
!grep -n "class GetoutActions" -A 8 env_src/getout/getout/actions.py
!python train.py -c in/config/getout.yaml -g getout 2>&1 | tail -15

4:class GetoutActions(Enum):
5-
6-    NOOP = 0
7-    MOVE_LEFT = 1
8-    MOVE_RIGHT = 2
9-    MOVE_UP = 3
10-    MOVE_DOWN = 4
11-
12-


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
!python train.py -g getout 2>&1 | grep "Not a directory"
!ls -la env_src/getout/getout/
!ls -la env_src/getout/getout/assets 2>/dev/null | head

NotADirectoryError: [Errno 20] Not a directory: '/content/NUDGE/env_src/getout/getout/getout.py/../../assets/kenney/Players/128x256/Green/alienGreen_stand.png'
total 144
drwxr-xr-x 3 root root  4096 Nov 14 18:51 .
drwxr-xr-x 5 root root  4096 Nov 14 18:51 ..
-rw-r--r-- 1 root root  1600 Nov 14 18:30 actions.py
-rw-r--r-- 1 root root   613 Nov 14 18:30 block.py
-rw-r--r-- 1 root root   547 Nov 14 18:30 boundingbox.py
-rw-r--r-- 1 root root  1188 Nov 14 18:30 camera.py
-rw-r--r-- 1 root root  1678 Nov 14 18:30 coin.py
-rw-r--r-- 1 root root   880 Nov 14 18:30 door.py
-rw-r--r-- 1 root root  1793 Nov 14 18:30 dummyGenerator.py
-rw-r--r-- 1 root root   261 Nov 14 18:30 entityEncoding.py
-rw-r--r-- 1 root root  8441 Nov 14 18:30 entity.py
-rw-r--r-- 1 root root   965 Nov 14 18:30 flag.py
-rw-r--r-- 1 root root  3130 Nov 14 18:30 getout.py
-rw-r--r-- 1 root root  2172 Nov 14 18:30 groundEnemy2.py
-rw-r--r-- 1 root root 13885 Nov 14 18:30 groundEnemy.py
-rw-r--r-- 1 root root   628 Nov 14 18:

In [3]:
!ls /content/NUDGE/env_src/getout/assets/kenney | head

Backgrounds
Enemies
Ground
HUD
Items
Particles
Players
Tiles


In [4]:
!sed -i '1i import os' env_src/getout/getout/resource_loader.py
!sed -i 's/Image.open(self._path.joinpath(rel_path))/Image.open(os.path.normpath(self._path.joinpath(rel_path)))/' env_src/getout/getout/resource_loader.py
!grep -n "normpath" env_src/getout/getout/resource_loader.py

54:        with Image.open(os.path.normpath(self._path.joinpath(rel_path))) as img:


In [5]:
!sed -i 's/gymnasium.make("getout")/gymnasium.envs.registration.load_env_creator(gymnasium.spec("getout").entry_point)(**gymnasium.spec("getout").kwargs)/' in/envs/getout/env.py
!grep -n "load_env_creator" in/envs/getout/env.py

30:        self.env = gymnasium.envs.registration.load_env_creator(gymnasium.spec("getout").entry_point)(**gymnasium.spec("getout").kwargs)  # FIXME


In [3]:
%cd /content/NUDGE/nsfr
!pip install -e .
%cd /content/NUDGE/nudge
!pip install -e .
%cd /content/NUDGE

/content/NUDGE/nsfr
Obtaining file:///content/NUDGE/nsfr
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nsfr (pyproject.toml) ... done
  Created wheel for nsfr: filename=nsfr-0.1.0-0.editable-py3-none-any.whl size=2965 sha256=e01ca8a7fad44d1b468bcc916b541dbe1d582e39813f73ac4ee98c2b9177bf14
  Stored in directory: /tmp/pip-ephem-wheel-cache-pdrrf0w8/wheels/ff/69/d4/41f4f147831c57d425b196c976e9190255a84ca89d10392021
Successfully built nsfr
/content/NUDGE/nudge
Obtaining file:///content/NUDGE/nudge
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-no

In [2]:
!pip uninstall -y tensorflow jax jaxlib

Found existing installation: tensorflow 2.21.0
Uninstalling tensorflow-2.21.0:
  Successfully uninstalled tensorflow-2.21.0
Found existing installation: jax 0.11.1
Uninstalling jax-0.11.1:
  Successfully uninstalled jax-0.11.1
Found existing installation: jaxlib 0.11.1
Uninstalling jaxlib-0.11.1:
  Successfully uninstalled jaxlib-0.11.1
